In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import statsmodels.formula.api as smf
import statsmodels.api as sm

from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, roc_curve, recall_score, average_precision_score
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix, precision_score, hamming_loss



from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn

In [2]:
import pickle
with open('balanced_data.pkl', 'rb') as b:
    data = pickle.load(b)

X_res_sk = data['X_res_stroke']
y_res_sk = data['y_res_stroke']
X_res_db = data['X_res_diab']
y_res_db = data['y_res_diab']
X_res_cn = data['X_res_can']
y_res_cn = data['y_res_can']       
X_res_ha = data['X_res_ha']
y_res_ha = data['y_res_ha']    

X_val = data['X_val'] 
y_val = data['y_val']   

X_test = data['X_test']
y_test = data['y_test']

X_bal=data['X_bal']
y_bal=data['y_bal']
    

In [3]:
y_test=y_test.rename(columns=
              {'heart_attack' : 'HA'})
y_bal=y_bal.rename(columns=
              {'heart_attack' : 'HA'})
y_val=y_val.rename(columns=
              {'heart_attack' : 'HA'})

***Creating a Dictionary containing all the values balanced***

In [ ]:
single_data = {
    'stroke' : (X_res_sk,y_res_sk),
    'diabetes' : (X_res_db,y_res_db),
    'cancer' : (X_res_cn,y_res_cn),
    'HA' : (X_res_ha,y_res_ha)
}


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
vif_results = {}

for outcome, (X_res, y_res) in single_data.items():
    
    vif = pd.DataFrame()
    vif["feature"] = X_res.columns
    vif["VIF"] = [variance_inflation_factor(X_res.values, i) 
                  for i in range(X_res.shape[1])]
    
    vif_results[outcome] = vif.sort_values("VIF", ascending=False)

print(vif_results['cancer'])

       feature         VIF
21         thb  180.318923
16        chol  114.757441
20         hdl  113.117471
22         a1c  104.852052
15         age   75.294543
19      cyst_c   67.731982
13         bmi   42.406370
14     maxgrip   24.917032
4      phactiv    7.459225
18       trigl    4.771695
11      gender    3.837262
10     alcohol    2.286487
5     drug_hbp    1.842260
3      fatigue    1.778179
0         depr    1.717555
17         crp    1.380123
9     drug_oth    1.254252
12       astar    1.245561
2       concen    1.237157
1     appetite    1.157610
6    drug_stom    1.152926
7   drug_sleep    1.137989
8     drug_inf    1.040055


In [6]:
cat = ['depr','appetite','concen','fatigue','phactiv',
       'drug_hbp','drug_stom','drug_sleep','drug_inf','drug_oth','alcohol','gender','astar']
num = ['bmi','maxgrip','age','chol','crp','trigl','cyst_c','hdl','thb','a1c']

In [7]:
print(single_data.keys())

dict_keys(['stroke', 'diabetes', 'cancer', 'HA'])


***LOGISTIC REGRESSION*** //
1 - One label at a time (X_res_* data used)
2 - Multilabel Logistic Regression (X_bal data used)

***ONE LABEL***

In [ ]:
np.random.seed(1234)

C_grid = np.logspace(1, 4, 20)
formula_bs = ' + '.join(cat + num)

single_models = {}
single_scalers = {}
single_data_scaled = {}

from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings("ignore")

for outcome, (X_res, y_res) in single_data.items():

    scaler = StandardScaler()

    X_res_scaled = X_res.copy()
    X_res_scaled[num] = scaler.fit_transform(X_res[num])

    single_data_scaled[outcome] = (X_res_scaled, y_res)
    single_scalers[outcome] = scaler

    grid = GridSearchCV(
        estimator=LogisticRegression(penalty="l2", max_iter=1000),
        param_grid={"C": C_grid},
        scoring="recall",
        cv=5,
        n_jobs=-1
    )

    grid.fit(X_res_scaled, y_res)

    best_C = grid.best_params_["C"]
    best_lambda = 1 / best_C

    print(f"C = {best_C} for {outcome}")
    print(f"for {outcome} the best lambda is {best_lambda}")

    train_data = X_res_scaled.copy()
    train_data[outcome] = y_res.values

    formula = f"{outcome} ~ " + formula_bs

    model_lg = smf.glm(
        formula=formula,
        data=train_data,
        family=sm.families.Binomial()
    ).fit_regularized(
        alpha=best_lambda,
        L1_wt=0
    )

    single_models[outcome] = model_lg

    coef_table = pd.DataFrame({
        "coef": model_lg.params,
        "odds_ratio": np.exp(model_lg.params)
    })

    print(coef_table)

C = 29.76351441631318 for stroke
for stroke the best lambda is 0.033598182862837826
                coef  odds_ratio
Intercept   0.307126    1.359513
depr       -0.258899    0.771901
appetite   -0.300475    0.740467
concen     -0.129067    0.878915
fatigue    -0.025083    0.975229
phactiv    -0.635084    0.529891
drug_hbp    0.259987    1.296913
drug_stom  -0.242925    0.784330
drug_sleep -0.158563    0.853370
drug_inf   -0.140204    0.869181
drug_oth   -0.505799    0.603024
alcohol    -0.486920    0.614516
gender      0.012826    1.012908
astar      -0.545507    0.579548
bmi         0.090123    1.094308
maxgrip    -0.111309    0.894662
age         0.316696    1.372585
chol       -0.251772    0.777422
crp        -0.007038    0.992987
trigl      -0.058352    0.943318
cyst_c      0.201995    1.223842
hdl         0.016109    1.016239
thb         0.112221    1.118760
a1c         0.030436    1.030904
C = 183.29807108324357 for diabetes
for diabetes the best lambda is 0.00545559478116852
   

In [ ]:
results_sLog={}
for out, (X_res, y_res) in single_data.items():
    y_prob = single_models[out].predict(X_test)  
    y_pred = (y_prob > 0.5).astype(int)          
    
    results_sLog[out] = {
        'f1': f1_score(y_test[out], y_pred),
        'precision': precision_score(y_test[out], y_pred),
        'sens': recall_score(y_test[out], y_pred),
        'auc':  roc_auc_score(y_test[out], y_prob),
        'auc-pr': average_precision_score(y_test[out], y_prob),
        'confusion':        confusion_matrix(y_test[out], y_pred),
        'misc': (y_pred != y_test[out].values).mean()
    }

print(pd.DataFrame({k: v for k, v in results_sLog.items()}).T)


                f1 precision      sens       auc    auc-pr  \
stroke         0.0       0.0       0.0   0.59027  0.048965   
diabetes  0.107728  0.234694  0.069909  0.655678  0.213999   
cancer    0.038674  0.042424  0.035533  0.554103   0.05051   
HA             0.0       0.0       0.0  0.609699  0.168007   

                         confusion      misc  
stroke       [[4457, 0], [154, 0]]  0.033398  
diabetes  [[3803, 150], [612, 46]]  0.165257  
cancer     [[4256, 158], [190, 7]]  0.075472  
HA           [[4063, 0], [548, 0]]  0.118846  


***MULTILABEL LOGISTIC REGRESSION*** 

In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

np.random.seed(1234)

C_grid_ml = np.logspace(1, 4, 20)  # C = 1/lambda

scaler_ml = StandardScaler()

X_bal_scaled = X_bal.copy()
X_bal_scaled[num] = scaler_ml.fit_transform(X_bal[num])

multigrid = GridSearchCV(
    estimator=MultiOutputClassifier(
        LogisticRegression(penalty="l2", max_iter=1000)
    ),
    param_grid={"estimator__C": C_grid_ml},
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

multigrid.fit(X_bal_scaled, y_bal)

multimodel = multigrid.best_estimator_

best_C_ml = multigrid.best_params_["estimator__C"]
best_lambda_ml = 1 / best_C_ml

print(multimodel)
print(f"C comune = {best_C_ml}")
print(f"lambda comune = {best_lambda_ml}")

MultiOutputClassifier(estimator=LogisticRegression(C=np.float64(10.0),
                                                   max_iter=1000,
                                                   penalty='l2'))
C comune = 10.0
lambda comune = 0.1


In [11]:
best_C = multigrid.best_params_['estimator__C']
lambda_common = 1 / best_C

print("C comune:", best_C)
print("lambda comune:", lambda_common)

C comune: 10.0
lambda comune: 0.1


In [12]:
target = y_bal.columns.tolist()
y_pred_ml =multigrid.predict(X_test)
y_prob_ml = multigrid.predict_proba(X_test)
results_ml = {}
for i, outcome in enumerate(target):
    y_true_i = y_test[outcome].values
    y_pred_i = y_pred_ml[:, i]
    y_prob_i = y_prob_ml[i][:, 1]

    results_ml[outcome] = {
        'f1': f1_score(y_true_i, y_pred_i),
        'auc': roc_auc_score(y_true_i, y_prob_i),
        'auc-pr': average_precision_score(y_true_i, y_prob_i),
        'misc': (y_pred_i != y_true_i).mean(),
        'sens': recall_score(y_true_i, y_pred_i),
        'confusion': confusion_matrix(y_true_i, y_pred_i),
        'precision': precision_score(y_true_i, y_pred_i)
    }
    
print(pd.DataFrame({k: v for k, v in results_ml.items()}).T)


                f1       auc    auc-pr      misc      sens  \
diabetes  0.044138  0.656011  0.213645  0.150293  0.024316   
stroke         0.0  0.604569  0.048111  0.033398       0.0   
HA             0.0  0.654969  0.185105  0.118846       0.0   
cancer     0.08037  0.486574  0.042615  0.947951  0.969543   

                        confusion precision  
diabetes  [[3902, 51], [642, 16]]  0.238806  
stroke      [[4457, 0], [154, 0]]       0.0  
HA          [[4063, 0], [548, 0]]       0.0  
cancer     [[49, 4365], [6, 191]]  0.041923  


In [13]:

print("y_bal proporzioni:\n", y_bal.mean())
print("y_test proporzioni:\n", y_test.mean())


y_bal proporzioni:
 diabetes    0.132913
stroke      0.090609
HA          0.118403
cancer      0.136035
dtype: float64
y_test proporzioni:
 diabetes    0.142702
stroke      0.033398
HA          0.118846
cancer      0.042724
dtype: float64
